# Crypto · arquitectura medallion sin Microsoft Fabric

**Que responde este cuaderno**

1. Cuanto se movio el mercado en el periodo cubierto (precio, volumen, retorno diario).
2. Cuantas monedas de las analizadas terminaron en positivo.
3. Si la suba es generalizada o la sostienen unas pocas.
4. Que falla en la capa de origen del pipeline y como se detecta.

**Datos**: dos tablas del repositorio (`crypto_monthly_summary` y `crypto_yearly_performance`) y un tercer archivo que el proyecto documenta como vacio. Se descargan por URL al ejecutar.

**Como correrlo**: Runtime > Run all. Solo pandas y matplotlib.

**Escala**: 13 meses y 100 monedas. Es una foto del periodo, no una serie larga.

In [ ]:
import io, pathlib, urllib.parse, urllib.request
import pandas as pd
import matplotlib.pyplot as plt

BASE = 'https://raw.githubusercontent.com/Nicolenki7/Crypto_Medallion_Analytics/main/data/'
pathlib.Path('outputs').mkdir(exist_ok=True)

def leer(nombre):
    with urllib.request.urlopen(BASE + urllib.parse.quote(nombre)) as r:
        return pd.read_csv(io.BytesIO(r.read()))

mensual = leer('crypto_monthly_summary - copia.csv')
anual = leer('crypto_yearly_performance - copia.csv')
mensual['month'] = pd.to_datetime(mensual['month'] + '-01')

print('mensual:', mensual.shape, list(mensual.columns))
print('anual  :', anual.shape, list(anual.columns))
print('periodo mensual:', mensual['month'].min().date(), '->', mensual['month'].max().date())

## 1. Como se movio el mercado

Precio medio mensual, volumen negociado y retorno diario medio del periodo.

In [ ]:
precio_medio = mensual['avg_price'].mean()
volumen_total = mensual['total_volume'].sum()
retorno_medio = mensual['avg_daily_return'].mean()
primer = mensual['avg_price'].iloc[0]
ultimo = mensual['avg_price'].iloc[-1]
var_precio = (ultimo / primer - 1) * 100

print(f'RESPUESTA: en {len(mensual)} meses el precio medio pasa de {primer:,.0f} a {ultimo:,.0f} USD ({var_precio:+.1f} %), con un retorno diario medio de {retorno_medio:+.2f} % y {volumen_total:,.2e} de volumen acumulado.')

ax = mensual.set_index('month')['avg_price'].plot(kind='line', marker='o', figsize=(9, 4))
ax.set_title('Precio medio mensual (USD)')
ax.set_ylabel('avg_price')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('outputs/precio_mensual.png', dpi=150)
plt.show()

## 2. Cuantas monedas ganaron

Con 100 monedas se puede medir amplitud: si sube casi todo, es un ciclo; si suben 20, es rotacion.

In [ ]:
positivas = (anual['total_return'] > 0).sum()
negativas = (anual['total_return'] < 0).sum()
n = len(anual)

print(f'RESPUESTA: {positivas} de {n} monedas ({positivas / n * 100:.0f} %) terminan el periodo en positivo y {negativas} en negativo.')

orden = anual.sort_values('total_return', ascending=False).reset_index(drop=True)
print('Mejor:', orden.iloc[0]['coin_name'], f"{orden.iloc[0]['total_return']:,.1f} %")
print('Peor :', orden.iloc[-1]['coin_name'], f"{orden.iloc[-1]['total_return']:,.1f} %")

ax = orden.head(10).set_index('coin_name')['total_return'].plot(kind='barh', figsize=(9, 4))
ax.invert_yaxis()
ax.set_title('Top 10 por retorno del periodo (%)')
ax.set_xlabel('total_return (%)')
plt.tight_layout()
plt.savefig('outputs/top_retornos.png', dpi=150)
plt.show()

## 3. Suba generalizada o de pocas

La mediana inmune a outliers: si la media supera mucho a la mediana, el resultado lo cargan unas pocas monedas.

In [ ]:
media = anual['total_return'].mean()
mediana = anual['total_return'].median()
top10 = anual.nlargest(10, 'total_return')

print(f'RESPUESTA: media {media:,.1f} % contra mediana {mediana:,.1f} %: la moneda tipica gana bastante menos que el promedio, porque el retorno lo empujan unos pocos casos extremos.')
print(f'La mejor decena de monedas suma {top10["total_return"].sum():,.1f} % de retorno frente a {anual["total_return"].sum():,.1f} % de todo el universo.')

print()
print('Monedas con el mejor retorno:')
print(top10[['coin_name', 'start_price', 'end_price', 'total_return']].to_string(index=False))
print()
print('y las peores:')
print(orden.tail(5)[['coin_name', 'total_return']].to_string(index=False))

## 4. Que falla en la capa de origen

El proyecto documenta un archivo historico vacio. Un pipeline que no distingue 'vacio' de 'error' propaga nulos silenciosamente.

In [ ]:
with urllib.request.urlopen(BASE + 'crypto_historical_365days.csv') as r:
    crudo = r.read()

try:
    historico = pd.read_csv(io.BytesIO(crudo))
    estado = f'legible, {historico.shape[0]} filas'
except pd.errors.EmptyDataError:
    historico = None
    estado = 'vacio: no tiene cabecera ni filas'

print(f'RESPUESTA: crypto_historical_365days.csv pesa {len(crudo)} bytes ({estado}). El cuaderno sigue con las tablas que si tienen datos y deja constancia del fallo en lugar de rellenarlo con ceros.')
print('Control que deberia tener el pipeline: si el archivo de origen pesa 0 bytes, la carga falla y alerta; no continua con un DataFrame vacio.')

## Limites

- 13 meses y 100 monedas: alcanza para describir el periodo, no para inferir tendencia estructural.
- `avg_price` es un promedio del mercado, no el precio de una moneda concreta.
- Los retornos anuales comparan precio inicial y final: no incluyen comisiones ni liquidez.
- El archivo historico esta vacio en el origen; este cuaderno lo documenta, no lo inventa.